# Smart City-project van Nova Stad

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, lit, udf, pandas_udf, PandasUDFType
from pyspark.sql.types import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, OneHotEncoder, Imputer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator


### Data inladen

In [0]:
%sh
unzip -q /Volumes/workspace/default/course_files/bdd/bdd:.zip -d /Volumes/workspace/default/course_files/bdd/

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/course_files/bdd/bdd:/"))

In [0]:
# unzip images
dbutils.fs.cp(
    "dbfs:/Volumes/workspace/default/course_files/bdd/bdd100k_images_10k.zip",
    "file:/tmp/bdd_images.zip"
)

dbutils.fs.cp(
    "dbfs:/Volumes/workspace/default/course_files/bdd/bdd100k_labels.zip",
    "file:/tmp/bdd_labels.zip"
)

In [0]:
import zipfile

zip_path = "/Volumes/workspace/default/course_files/bdd100k_images_10k.zip"
extract_path = "/Volumes/workspace/default/course_files/bdd/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [0]:
df_traffic = spark.read.csv("/FileStore/tables/metro_interstate_traffic_volume_train.csv", header=True, inferSchema=True)

In [0]:
df = spark.read.csv(
    "/Volumes/main/default/datasets/metro_interstate_traffic_volume_train.csv",
    header=True,
    inferSchema=True
)

display(df.head(5))

In [0]:
# Benodigde libraries importeren
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, count, to_timestamp, year, month, dayofmonth, hour, dayofweek, regexp_replace
from pyspark.sql.types import StringType

# Spark session starten of ophalen
spark = SparkSession.builder.appName("MetroTrafficCleaning").getOrCreate()

In [0]:
# Pad naar train- en testbestand
train_path = "file:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
test_path  = "file:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"

In [0]:
# Pad naar train- en testbestand
train_path = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
test_path  = "dbfs:/Workspace/Repos/20076436@student.hhs.nl/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"

# Trainbestand inladen
df_train = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(train_path)
)

# Testbestand inladen
df_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(test_path)
)

# Eerste rijen tonen
display(df_train)
display(df_test)

In [0]:
# Aantal rijen en kolommen bekijken
print("Train rows:", df_train.count())
print("Train columns:", len(df_train.columns))

print("Test rows:", df_test.count())
print("Test columns:", len(df_test.columns))

# Schema bekijken
df_train.printSchema()
df_test.printSchema()

In [0]:
# Dubbele rijen verwijderen
df_train = df_train.dropDuplicates()
df_test = df_test.dropDuplicates()

# Controle na verwijderen
print("Train rows na dropDuplicates:", df_train.count())
print("Test rows na dropDuplicates:", df_test.count())

In [0]:
# Tekstkolommen zoeken
string_cols_train = [field.name for field in df_train.schema.fields if isinstance(field.dataType, StringType)]
string_cols_test = [field.name for field in df_test.schema.fields if isinstance(field.dataType, StringType)]

print("String kolommen train:", string_cols_train)
print("String kolommen test:", string_cols_test)

In [0]:
# Spaties voor/achter en dubbele spaties in tekst verwijderen
for c in string_cols_train:
    df_train = df_train.withColumn(c, trim(col(c)))
    df_train = df_train.withColumn(c, regexp_replace(col(c), r"\s+", " "))

for c in string_cols_test:
    df_test = df_test.withColumn(c, trim(col(c)))
    df_test = df_test.withColumn(c, regexp_replace(col(c), r"\s+", " "))

In [0]:
# Datumkolom omzetten naar timestamp
df_train = df_train.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))
df_test = df_test.withColumn("date_time", to_timestamp(col("date_time"), "yyyy-MM-dd HH:mm:ss"))

# Schema opnieuw bekijken
df_train.printSchema()
df_test.printSchema()

In [0]:
# Aantal missende waardes per kolom tellen
display(
    df_train.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_train.columns
    ])
)

display(
    df_test.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_test.columns
    ])
)

In [0]:
# Numerieke kolommen bepalen
numeric_types = ["int", "bigint", "double", "float", "long", "short", "decimal"]

numeric_cols_train = [f.name for f in df_train.schema.fields if f.dataType.simpleString() in numeric_types]
numeric_cols_test = [f.name for f in df_test.schema.fields if f.dataType.simpleString() in numeric_types]

print("Numerieke kolommen train:", numeric_cols_train)
print("Numerieke kolommen test:", numeric_cols_test)

In [0]:
# Nulls in numerieke kolommen vullen met mediaan
for c in numeric_cols_train:
    median_value = df_train.approxQuantile(c, [0.5], 0.01)[0]
    df_train = df_train.fillna({c: median_value})

for c in numeric_cols_test:
    median_value = df_test.approxQuantile(c, [0.5], 0.01)[0]
    df_test = df_test.fillna({c: median_value})

In [0]:
# Nulls in tekstkolommen vullen met "Unknown"
for c in string_cols_train:
    df_train = df_train.fillna({c: "Unknown"})

for c in string_cols_test:
    df_test = df_test.fillna({c: "Unknown"})

In [0]:
# Extra tijdkolommen maken uit date_time
df_train = (
    df_train
    .withColumn("year", year(col("date_time")))
    .withColumn("month", month(col("date_time")))
    .withColumn("day", dayofmonth(col("date_time")))
    .withColumn("hour", hour(col("date_time")))
    .withColumn("dayofweek", dayofweek(col("date_time")))
)

df_test = (
    df_test
    .withColumn("year", year(col("date_time")))
    .withColumn("month", month(col("date_time")))
    .withColumn("day", dayofmonth(col("date_time")))
    .withColumn("hour", hour(col("date_time")))
    .withColumn("dayofweek", dayofweek(col("date_time")))
)

In [0]:
# Schone data bekijken
display(df_train.limit(5))
display(df_test.limit(5))